# 05 — Diagnostiquer et corriger le biais envers l’écriture latine

Le notebook 04 obtenait un bon macro-F1 global sur dev, mais son rappel des négatifs en écriture latine était nul (0/34). Le test contient davantage de cette écriture et des mentions absentes du train. Ce constat ne prouve pas que les annotations sont erronées.

**Décision : conserver les données et corriger la méthodologie.** Nous comparons un témoin et des variantes de nettoyage, de régularisation et de pondération. Aucun texte ni label officiel n’est modifié sur disque.

La sélection utilise exclusivement une validation interne de 20 % du train, stratifiée par label et écriture. Le vocabulaire et l’IDF sont appris sur les 80 % restants. Dev et test servent ensuite au bilan. Le test a déjà été consulté lors des expériences précédentes : il ne constitue plus une évaluation indépendante de notre démarche.

Exécuter sur le cluster Docker, depuis PowerShell :

```powershell
powershell.exe -NoProfile -ExecutionPolicy Bypass -File .\scripts\Invoke-SparkNotebook.ps1 -Notebook notebooks/05_sentiment_script_aware.ipynb
```

Le runner exécute directement les cellules, sans exporter de fichier Python. Les résultats datés sont écrits sous `data/processed/afrisenti/diagnostics/` et dans HDFS.

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
import csv
import hashlib
import json
import math
import time

from pyspark import StorageLevel
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import CountVectorizer, IDF, NGram, Normalizer, RegexTokenizer, SQLTransformer, VectorAssembler
from pyspark.sql import SparkSession, Window, functions as F, types as T

INPUT_DIR = Path("/opt/lahjalens/data/processed/afrisenti/v1")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
LOCAL_DIR = INPUT_DIR.parent / "diagnostics" / RUN_ID
HDFS_DIR = "hdfs://namenode:9000/data/lahjalens/models/afrisenti_script_v3/" + RUN_ID
OLD04_PATH = "hdfs://namenode:9000/data/lahjalens/models/afrisenti_tfidf_v2/20261115T022358Z_feec1d32/model"
OLD03_PATH = "hdfs://namenode:9000/data/lahjalens/models/afrisenti_v1/experiments/20261115T014515Z/lr_word_char"
LABEL_NAMES = ["negative", "neutral", "positive"]
SCRIPTS = ["arabic", "latin"]
FIELDS = ["record_id", "comment_text", "sentiment", "label", "source_dataset",
          "source_split", "language", "cleaning_version"]

spark = (SparkSession.builder.appName("LahjaLens-Script-Aware-" + RUN_ID)
         .master("spark://spark-master:7077")
         .config("spark.hadoop.dfs.replication", "2")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.default.parallelism", "4")
         .config("spark.sql.csv.parser.columnPruning.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
assert spark.sparkContext.master == "spark://spark-master:7077"
spark.sparkContext._jsc.hadoopConfiguration().set("dfs.replication", "2")
LOCAL_DIR.mkdir(parents=True, exist_ok=False)

def write_json(name, value):
    (LOCAL_DIR / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")

report = dict(run_id=RUN_ID, status="running", dataset="afrisenti_ary",
              spark_application_id=spark.sparkContext.applicationId,
              master=spark.sparkContext.master, spark_version=spark.version,
              notebook_sha256=hashlib.sha256(Path("/opt/lahjalens/notebooks/05_sentiment_script_aware.ipynb").read_bytes()).hexdigest(),
              historical_test_already_consulted=True,
              selection="internal_train_holdout.mean_script_macro_f1; dev/test excluded",
              results=[])
print("RUN_ID", RUN_ID, flush=True)

## 1. Audit reproductible

On vérifie les empreintes, la lecture CSV, les labels, les doublons et les recouvrements entre splits. L’écriture dominante est une heuristique de comptage de caractères, **pas un détecteur de langue** : la catégorie latine peut inclure du français et du code-switching.

Le nettoyage retire les mentions et les URL, harmonise les variantes de l’alif, enlève les voyelles diacritiques et le tatweel, réduit les lettres répétées à deux et compacte les espaces. Il conserve les chiffres Arabizi, la négation et les emojis. Ces opérations seront enregistrées dans le PipelineModel pour être identiques en prédiction.

In [ ]:
def regex_sql(expression, pattern, replacement):
    assert "'" not in pattern and "'" not in replacement
    return "regexp_replace(" + expression + ", r'" + pattern + "', '" + replacement + "')"

def clean_expression(column):
    expression = "lower(" + column + ")"
    for pattern, replacement in [
        (r"https?://\S+|www\.\S+", " "),
        (r"@[\w]+", " "),
        ("[\u0640\u064b-\u065f\u0670]", ""),
        ("[\u0622\u0623\u0625\u0671]", "\u0627"),
        (r"([a-z\u0621-\u064a])\1{2,}", "$1$1"),
        (r"\s+", " "),
    ]:
        expression = regex_sql(expression, pattern, replacement)
    return "trim(" + expression + ")"

CLEAN_SQL = "SELECT *, " + clean_expression("comment_text") + " AS model_text FROM __THIS__"
cleaner = SQLTransformer(statement=CLEAN_SQL)
probe = cleaner.transform(spark.createDataFrame(
    [("@user salammmm 7na machi https://t.co/abc",), ("\u0623\u0646\u0627 \u0645\u0627\u0634\u064a \u062e\u0627\u064a\u0628",)], ["comment_text"])).collect()
assert probe[0]["model_text"] == "salamm 7na machi"
assert probe[1]["model_text"] == "\u0627\u0646\u0627 \u0645\u0627\u0634\u064a \u062e\u0627\u064a\u0628"

audit = json.loads((INPUT_DIR / "audit.json").read_text(encoding="utf-8"))
schema = T.StructType([T.StructField(name, T.StringType(), True) for name in FIELDS])
frames, input_audit, normalized_keys = {}, {}, {}
for split in ("train", "dev", "test"):
    path = INPUT_DIR / (split + ".csv")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == audit["splits"][split]["sha256"]
    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, strict=True)
        assert reader.fieldnames == FIELDS
        rows = list(reader)
    assert len(rows) == audit["splits"][split]["rows"]
    assert len({r["record_id"] for r in rows}) == len(rows)
    assert all(r["label"] == str(LABEL_NAMES.index(r["sentiment"])) for r in rows)
    frame = (spark.read.schema(schema).options(
        header=True, enforceSchema=False, mode="FAILFAST", encoding="UTF-8",
        quote='"', escape='"', multiLine=True).csv(str(path)))
    assert {r["record_id"]: r.asDict() for r in frame.collect()} == {r["record_id"]: r for r in rows}
    # Same transparent heuristic for all historical and new models.
    arabic_count = F.length(F.regexp_replace("comment_text", r"[^\u0600-\u06ff]", ""))
    latin_count = F.length(F.regexp_replace("comment_text", "[^A-Za-z]", ""))
    frame = (frame.withColumn("label", F.col("label").cast("double"))
             .withColumn("writing_script", F.when(arabic_count > latin_count, "arabic").otherwise("latin")).cache())
    frames[split] = frame
    keys = cleaner.transform(frame).select("model_text", "label").collect()
    normalized_keys[split] = {r["model_text"] for r in keys}
    assert "" not in normalized_keys[split]
    assert len(normalized_keys[split]) == len(keys), "Inspect normalization duplicates before training"
    input_audit[split] = dict(
        rows=len(rows), sha256=digest,
        labels=dict(Counter(r["sentiment"] for r in rows)),
        script_label_counts=[r.asDict() for r in frame.groupBy("writing_script", "sentiment").count().orderBy("writing_script", "sentiment").collect()],
        mention_rows=frame.where(F.col("comment_text").contains("@")).count())
for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
    assert not normalized_keys[a] & normalized_keys[b], "Normalized text overlap: " + a + "/" + b

# Fixed hash ordering within each of the six strata, independent of input order.
rank_window = Window.partitionBy("writing_script", "label").orderBy(
    F.sha2(F.concat(F.lit("lahjalens-v3-seed-20261115:"), F.col("comment_text")), 256), "record_id")
ranked = frames["train"].withColumn("_rank", F.row_number().over(rank_window))
fit_train = ranked.where(F.col("_rank") % 5 != 0).drop("_rank").cache()
inner = ranked.where(F.col("_rank") % 5 == 0).drop("_rank").cache()
assert fit_train.count() + inner.count() == input_audit["train"]["rows"]
assert fit_train.select("record_id").intersect(inner.select("record_id")).count() == 0
assignment = (fit_train.select("record_id").withColumn("role", F.lit("fit"))
              .union(inner.select("record_id").withColumn("role", F.lit("validation"))))
with (LOCAL_DIR / "inner_split.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["record_id", "role"])
    writer.writeheader()
    writer.writerows(r.asDict() for r in assignment.orderBy("record_id").collect())
report["input_audit"] = input_audit
report["inner_split"] = dict(fit_rows=fit_train.count(), validation_rows=inner.count(),
    strata=[r.asDict() for r in inner.groupBy("writing_script", "sentiment").count().collect()],
    assignment_sha256=hashlib.sha256((LOCAL_DIR / "inner_split.csv").read_bytes()).hexdigest())
write_json("input_audit.json", input_audit)
write_json("report.json", report)
print("AUDIT", json.dumps(input_audit), flush=True)

## 2. Comparaison contrôlée

- **Témoin** : caractéristiques du notebook 04, mais davantage d’itérations pour examiner la convergence.
- **Pondération classique** : poids inverse de la fréquence du sentiment.
- **Pondération par groupe** : poids inverse de la fréquence de chaque couple écriture–sentiment. Aucun exemple n’est dupliqué et aucune ligne du test n’est utilisée pour calculer les poids.
- **Représentations** : caractères de longueur 3 à 5, seuls ou combinés aux unigrammes/bigrammes de mots.
- **Sélection** : moyenne des macro-F1 des deux écritures sur le holdout interne. Un candidat qui ne reconnaît jamais une des six catégories observées est écarté.

La standardisation du classifieur et la normalisation L2 des documents sont deux opérations distinctes. Le paramètre de régularisation est fixé pour chaque candidat avant tout calcul des nouveaux scores test. L’historique de l’objectif permet de vérifier si la limite d’itérations a interrompu l’optimisation.

In [ ]:
def weighted(frame, mode):
    columns = ["label"] if mode == "class" else ["writing_script", "label"]
    counts = frame.groupBy(*columns).count()
    groups = counts.count()
    assert groups == (3 if mode == "class" else 6)
    n = frame.count()
    weights = counts.withColumn("sample_weight", F.lit(float(n) / groups) / F.col("count")).drop("count")
    result = frame.join(weights, columns, "inner")
    assert result.count() == n
    return result, [r.asDict() for r in weights.orderBy(*columns).collect()]

def feature_pipeline(representation):
    clean = representation != "raw_char"
    stages = [cleaner.copy() if clean else SQLTransformer(
        statement="SELECT *, lower(comment_text) AS model_text FROM __THIS__")]
    stages += [SQLTransformer(statement="""
        SELECT *, flatten(transform(array(3,4,5), n ->
          CASE WHEN length(model_text) >= n THEN
            transform(sequence(1, length(model_text)-n+1), i -> substring(model_text,i,n))
          ELSE cast(array() AS array<string>) END)) AS char_tokens FROM __THIS__
    """), CountVectorizer(inputCol="char_tokens", outputCol="char_tf", vocabSize=80000, minDF=2.0),
        IDF(inputCol="char_tf", outputCol="char_idf", minDocFreq=2),
        Normalizer(inputCol="char_idf", outputCol="char_norm" if representation=="clean_wordchar" else "features", p=2.0)]
    if representation == "clean_wordchar":
        stages += [RegexTokenizer(inputCol="model_text", outputCol="words", pattern=r"\s+", minTokenLength=1),
                   NGram(n=2, inputCol="words", outputCol="bigrams"),
                   SQLTransformer(statement="SELECT *, concat(words,bigrams) AS word_tokens FROM __THIS__"),
                   CountVectorizer(inputCol="word_tokens", outputCol="word_tf", vocabSize=50000, minDF=2.0),
                   IDF(inputCol="word_tf", outputCol="word_idf", minDocFreq=2),
                   Normalizer(inputCol="word_idf", outputCol="word_norm", p=2.0),
                   VectorAssembler(inputCols=["word_norm","char_norm"], outputCol="combined"),
                   Normalizer(inputCol="combined", outputCol="features", p=2.0)]
    return Pipeline(stages=stages)

def matrix_metrics(matrix):
    n = sum(map(sum, matrix))
    classes = {}
    for i, name in enumerate(LABEL_NAMES):
        support = sum(matrix[i])
        predicted = sum(row[i] for row in matrix)
        tp = matrix[i][i]
        classes[name] = dict(support=support, precision=tp/predicted if predicted else 0.0,
                             recall=tp/support if support else 0.0,
                             f1=2*tp/(support+predicted) if support+predicted else 0.0)
    return dict(rows=n, accuracy=sum(matrix[i][i] for i in range(3))/n if n else 0.0,
                macro_f1=sum(v["f1"] for v in classes.values())/3,
                weighted_f1=sum(v["f1"]*v["support"] for v in classes.values())/n if n else 0.0,
                per_class=classes, confusion_matrix=matrix, order=LABEL_NAMES)

def measure(predictions):
    matrices = {s: [[0]*3 for _ in range(3)] for s in SCRIPTS}
    for row in predictions.groupBy("writing_script","label","prediction").count().collect():
        i,j = int(row["label"]),int(row["prediction"])
        assert i in range(3) and j in range(3)
        matrices[row["writing_script"]][i][j] += row["count"]
    total = [[sum(matrices[s][i][j] for s in SCRIPTS) for j in range(3)] for i in range(3)]
    by_script = {s: matrix_metrics(matrices[s]) for s in SCRIPTS}
    return dict(overall=matrix_metrics(total), by_script=by_script,
                mean_script_macro_f1=sum(v["macro_f1"] for v in by_script.values())/2,
                minimum_group_recall=min(v["recall"] for s in by_script.values() for v in s["per_class"].values()))

def fit_classifier(vectors, setting):
    started = time.monotonic()
    classifier = LogisticRegression(family="multinomial", maxIter=300, tol=1e-6,
        regParam=setting["reg"], standardization=setting["standardization"], weightCol="sample_weight")
    fitted = classifier.fit(vectors)
    history = [float(v) for v in fitted.summary.objectiveHistory]
    assert history and all(math.isfinite(v) for v in history)
    return fitted, dict(iterations=int(fitted.summary.totalIterations),
        reached_max_iterations=int(fitted.summary.totalIterations)>=300,
        initial_objective=history[0], final_objective=history[-1],
        last_objectives=history[-10:], objective_history=history,
        seconds=round(time.monotonic()-started,2))

GRID = [
    dict(name="raw_char_convergence_control", representation="raw_char", weight="class", reg=0.001, standardization=False),
    dict(name="clean_char_class", representation="clean_char", weight="class", reg=0.1, standardization=True),
    dict(name="clean_char_group", representation="clean_char", weight="group", reg=0.1, standardization=True),
    dict(name="clean_char_group_nostd", representation="clean_char", weight="group", reg=0.001, standardization=False),
    dict(name="clean_wordchar_class", representation="clean_wordchar", weight="class", reg=0.1, standardization=True),
    dict(name="clean_wordchar_group", representation="clean_wordchar", weight="group", reg=0.1, standardization=True),
    dict(name="clean_wordchar_group_reg005", representation="clean_wordchar", weight="group", reg=0.05, standardization=True),
    dict(name="clean_wordchar_group_reg03", representation="clean_wordchar", weight="group", reg=0.3, standardization=True),
]
report["grid"] = GRID
write_json("selection_protocol.json", dict(grid=GRID, selection=report["selection"], test_used_for_selection=False,
    source_data_changed=False, cleaned_text_used_as_model_feature=True))
write_json("report.json", report)

In [ ]:
best = None
for representation in ("raw_char", "clean_char", "clean_wordchar"):
    fm = feature_pipeline(representation).fit(fit_train)
    fit_vectors = fm.transform(fit_train).select("record_id","writing_script","label","features").repartition(4).persist(StorageLevel.MEMORY_AND_DISK)
    val_vectors = fm.transform(inner).select("record_id","writing_script","label","features").repartition(4).persist(StorageLevel.MEMORY_AND_DISK)
    fit_vectors.count()
    val_vectors.count()
    for setting in [s for s in GRID if s["representation"]==representation]:
        weighted_vectors, weights = weighted(fit_vectors, setting["weight"])
        weighted_vectors = weighted_vectors.repartition(4).persist(StorageLevel.MEMORY_AND_DISK)
        weighted_vectors.count()
        fitted, convergence = fit_classifier(weighted_vectors, setting)
        metrics = measure(fitted.transform(val_vectors))
        eligible = metrics["minimum_group_recall"] > 0
        result = dict(setting, validation=metrics, eligible=eligible, weights=weights, convergence=convergence)
        report["results"].append(result)
        if eligible and (best is None or metrics["mean_script_macro_f1"] > best["validation"]["mean_script_macro_f1"]):
            best = result
        write_json("report.json", report)
        print("CANDIDATE", setting["name"], "macro", round(metrics["overall"]["macro_f1"],4),
              "script_macro", round(metrics["mean_script_macro_f1"],4),
              "min_recall", round(metrics["minimum_group_recall"],4),
              "iterations", convergence["iterations"], flush=True)
        weighted_vectors.unpersist()
    fit_vectors.unpersist()
    val_vectors.unpersist()
assert best is not None, "No candidate recognizes every observed script/label group"
report["selected_setting"] = {k: best[k] for k in ("name","representation","weight","reg","standardization")}
write_json("selection_locked.json", report["selected_setting"])
print("SELECTED_BEFORE_DEV_TEST", report["selected_setting"], flush=True)

# Refit only the locked setting on the entire official training split.
setting = report["selected_setting"]
full_train, full_weights = weighted(frames["train"], setting["weight"])
fm = feature_pipeline(setting["representation"]).fit(full_train)
full_vectors = fm.transform(full_train).select("features","label","sample_weight").repartition(4).persist(StorageLevel.MEMORY_AND_DISK)
full_vectors.count()
classifier, convergence = fit_classifier(full_vectors, setting)
model = PipelineModel(stages=fm.stages + [classifier])
report["full_fit"] = dict(weights=full_weights, convergence=convergence)
report["train"] = measure(model.transform(frames["train"]))
full_vectors.unpersist()
report["model_path"] = HDFS_DIR + "/model"
model.write().save(report["model_path"])
write_json("report.json", report)

## 3. Bilan final et comparaison sur les mêmes lignes

On recharge le modèle HDFS et on vérifie ses prédictions. Les anciens modèles 03 et 04 sont ensuite évalués sur les **mêmes fichiers CSV corrigés**, sans modifier la sélection. Le modèle 03 a besoin d’une transformation externe des trigrammes ; celle-ci reproduit sa préparation d’origine.

Les prédictions et matrices par écriture permettent de distinguer le gain moyen d’un déplacement des erreurs entre groupes. Un intervalle bootstrap apparié sur le test décrit la variabilité de la différence ; il n’annule pas le fait que ce test a été consulté auparavant.

Les scores sur AfriSenti ne valident pas le transfert vers Hespress ou Instagram. Une nouvelle évaluation humaine sur des commentaires de la source cible reste nécessaire.

In [ ]:
reloaded = PipelineModel.load(report["model_path"])
expected = model.transform(frames["dev"]).select("record_id", F.col("prediction").alias("expected"))
actual = reloaded.transform(frames["dev"]).select("record_id", "prediction")
assert expected.join(actual,"record_id").where(F.col("expected") != F.col("prediction")).count() == 0
report["reload_prediction_mismatches"] = 0

models = {"new": reloaded}
historical_skipped = {}
for key, path in (("notebook04",OLD04_PATH), ("notebook03",OLD03_PATH)):
    hpath = spark._jvm.org.apache.hadoop.fs.Path(path)
    if hpath.getFileSystem(spark._jsc.hadoopConfiguration()).exists(hpath):
        models[key] = PipelineModel.load(path)
    else:
        historical_skipped[key] = "Model not available in this HDFS installation: " + path
report["historical_skipped"] = historical_skipped
historical_chars = SQLTransformer(statement=r"""
SELECT *, CASE WHEN length(_legacy_text)>=3 THEN
 transform(sequence(1,length(_legacy_text)-2), i->substring(_legacy_text,i,3))
 ELSE cast(array() AS array<string>) END AS char_tokens
FROM (SELECT *, regexp_replace(comment_text,r'\s+','_') AS _legacy_text FROM __THIS__)
""")
report["evaluation"] = {}
prediction_rows = {}
for name, fitted in models.items():
    report["evaluation"][name] = {}
    for split in ("dev","test"):
        frame = frames[split]
        if name == "notebook03":
            frame = historical_chars.transform(frame)
        predictions = fitted.transform(frame).select("record_id","writing_script","label","prediction").cache()
        metrics = measure(predictions)
        assert metrics["overall"]["rows"] == input_audit[split]["rows"]
        report["evaluation"][name][split] = metrics
        rows = [r.asDict() for r in predictions.orderBy("record_id").collect()]
        prediction_rows[(name,split)] = rows
        with (LOCAL_DIR / (name+"_"+split+"_predictions.csv")).open("w",encoding="utf-8",newline="") as handle:
            writer = csv.DictWriter(handle,fieldnames=["record_id","writing_script","label","prediction"])
            writer.writeheader()
            writer.writerows(rows)
        if name == "new":
            predictions.write.mode("errorifexists").parquet(HDFS_DIR+"/predictions/"+split)
        predictions.unpersist()
        print("EVALUATION", name, split, json.dumps(metrics), flush=True)
    write_json("report.json", report)

# Paired bootstrap: same sampled rows for new and reference predictions.
import numpy as np
def array_macro(y, p):
    matrix = np.bincount(y*3+p, minlength=9).reshape(3,3)
    denom = matrix.sum(axis=0)+matrix.sum(axis=1)
    return float(np.divide(2*np.diag(matrix), denom, out=np.zeros(3), where=denom>0).mean())
report["paired_bootstrap"] = {}
new_rows = prediction_rows[("new","test")]
y = np.array([int(r["label"]) for r in new_rows])
p = np.array([int(r["prediction"]) for r in new_rows])
for name in ("notebook03","notebook04"):
    if (name,"test") not in prediction_rows:
        continue
    previous = prediction_rows[(name,"test")]
    assert [r["record_id"] for r in previous] == [r["record_id"] for r in new_rows]
    assert [r["label"] for r in previous] == [r["label"] for r in new_rows]
    q = np.array([int(r["prediction"]) for r in previous])
    rng = np.random.default_rng(20261115)
    differences = []
    for _ in range(2000):
        indices = rng.integers(0,len(y),size=len(y))
        differences.append(array_macro(y[indices],p[indices])-array_macro(y[indices],q[indices]))
    report["paired_bootstrap"][name] = dict(
        delta_macro_f1=array_macro(y,p)-array_macro(y,q),
        percentile_95=[float(v) for v in np.quantile(differences,[0.025,0.975])],
        repetitions=2000, seed=20261115, exploratory=True)

# No source CSV is changed by this notebook.
for split in ("train","dev","test"):
    assert hashlib.sha256((INPUT_DIR/(split+".csv")).read_bytes()).hexdigest() == input_audit[split]["sha256"]
report["status"] = "completed"
report["completed_at"] = datetime.now(timezone.utc).isoformat()
report["local_dir"] = str(LOCAL_DIR)
report["hdfs_dir"] = HDFS_DIR
write_json("report.json",report)
spark.createDataFrame([(json.dumps(report,ensure_ascii=False,allow_nan=False),)],["value"]).coalesce(1).write.text(HDFS_DIR+"/metrics")
print("COMPLETED", RUN_ID, "model",report["model_path"],"test_macro_f1",
      report["evaluation"]["new"]["test"]["overall"]["macro_f1"],flush=True)
spark.stop()